# 02 Traces with pandas
Clean, slice, sample, redact, and build a failure backlog.

Step 1: set up the kit

In [ ]:
# Setup: in Colab, clone the kit first. Locally, find the kit folder. Change REPO_URL to your fork.
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/GITHUB_OWNER/atp-builder-kit"
if "google.colab" in sys.modules and not Path("kit/llm.py").exists():
    if not Path("atp-builder-kit").exists():
        subprocess.run(["git", "clone", REPO_URL], check=True)
    os.chdir("atp-builder-kit")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pandas"], check=True)

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "kit" / "llm.py").exists())
sys.path.insert(0, str(ROOT))
DATA = Path(os.environ.get("KIT_DATA_DIR", ROOT / "data"))
OUTPUT = Path(os.environ.get("KIT_OUTPUT_DIR", ROOT / "stage-02"))
OUTPUT.mkdir(parents=True, exist_ok=True)
print("Kit folder:", ROOT)

Step 2: load the traces

In [ ]:
import re
import pandas as pd
raw = pd.read_json(DATA / "traces.jsonl", lines=True)
print(len(raw), "rows")
raw.head()

Step 3: drop duplicates, empty outputs and test traffic

In [ ]:
df = raw.drop_duplicates(subset="trace_id")
print("Duplicates removed:", len(raw) - len(df))
empty = df.assistant_output.str.strip() == ""
print("Empty outputs removed:", int(empty.sum()))
df = df[~empty]
test = df.user_id.str.startswith("test_")
print("Test traffic removed:", int(test.sum()))
df = df[~test].copy()
df["failed"] = df.outcome == "failed"
print(len(df), "rows left")

Step 4: failure rate by segment and by channel

In [ ]:
print(df.groupby("segment").failed.mean().round(3))
print(df.groupby("channel").failed.mean().round(3))
df.pivot_table(index="segment", columns="channel", values="failed", aggfunc="mean").round(3)

Step 5: a random sample and a stratified sample of 30

In [ ]:
random_30 = df.sample(30, random_state=7)
stratified_30 = df.groupby("segment").sample(n=10, random_state=7)
print("Random sample by segment:", random_30.segment.value_counts().to_dict())
print("Stratified sample by segment:", stratified_30.segment.value_counts().to_dict())
stratified_30[["trace_id", "segment", "channel", "user_message", "outcome"]].to_csv(OUTPUT / "labeling_sample.csv", index=False)

Step 6: redact PII with regex

In [ ]:
PATTERNS = {
    "card": r"\b(?:\d{4}[ -]?){3}\d{4}\b",
    "email": r"[\w.+-]+@[\w-]+\.[\w.]+",
    "phone": r"\+?\d{0,2}\s?\d{3}-\d{3}-\d{4}",
}

def redact(text):
    for name, pattern in PATTERNS.items():
        text = re.sub(pattern, f"[{name.upper()}]", text)
    return text

df["redacted"] = df.user_message.map(redact)
df[["user_message", "redacted"]].head(3)

Step 7: count caught and missed against the planted ground truth

In [ ]:
checks = [(value, value not in red) for values, red in zip(df.pii_planted, df.redacted) for value in values]
caught = sum(ok for _, ok in checks)
missed = [value for value, ok in checks if not ok]
print(f"Planted: {len(checks)} | caught: {caught} | missed: {len(missed)}")
print("Examples missed:", missed[:5])

Step 8: save a ranked failure backlog

In [ ]:
failures = df[df.failed].copy()
cohort_rate = df.groupby(["segment", "channel"]).failed.mean().rename("cohort_failure_rate")
failures = failures.join(cohort_rate, on=["segment", "channel"])
failures = failures.sort_values(["cohort_failure_rate", "latency_ms"], ascending=False)
backlog = failures[["trace_id", "segment", "channel", "cohort_failure_rate", "redacted"]].rename(columns={"redacted": "user_message"})
backlog.insert(0, "rank", range(1, len(backlog) + 1))
backlog["layer_guess"] = ""  # you fill this in: prompt, retrieval, model, tool, UI ...
backlog["severity"] = ""     # you fill this in
backlog.to_csv(OUTPUT / "failure_backlog.csv", index=False)
print("Saved", len(backlog), "rows to", OUTPUT / "failure_backlog.csv")
backlog.head(10)